# 🏋️ Module 5.5: Exercises

---

In [ ]:
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

mlflow.autolog(disable=True)
mlflow.set_experiment("05_exercises")

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("✅ Ready!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## Exercise 1: Log a Model with Signature

Train a RandomForestClassifier and log it with:
- An inferred signature
- An input example (first 3 rows)
- Accuracy metric

Then load it back and verify predictions work.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
with mlflow.start_run(run_name="ex1_signature"):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    sig = infer_signature(X_train, preds)
    mlflow.sklearn.log_model(model, "model", signature=sig, input_example=X_train[:3])
    mlflow.log_metric("accuracy", accuracy_score(y_test, preds))
    
    rid = mlflow.active_run().info.run_id

loaded = mlflow.pyfunc.load_model(f"runs:/{rid}/model")
result = loaded.predict(X_test[:5])
print(f"Predictions: {result}")
print("✅ Done!")

## Exercise 2: Build a Custom PyFunc

Create a custom PyFunc model that:
1. Scales input data with StandardScaler
2. Predicts with a RandomForest
3. Returns a DataFrame with columns: `prediction`, `class_name`, `confidence`

Log it to MLFlow and verify it loads correctly.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
class MyWineModel(mlflow.pyfunc.PythonModel):
    def __init__(self, scaler, model, names):
        self.scaler = scaler
        self.model = model
        self.names = names
    
    def predict(self, context, model_input, params=None):
        scaled = self.scaler.transform(model_input)
        preds = self.model.predict(scaled)
        probs = self.model.predict_proba(scaled)
        return pd.DataFrame({
            "prediction": preds,
            "class_name": [self.names[p] for p in preds],
            "confidence": [p.max() for p in probs]
        })

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, y_train)

custom = MyWineModel(scaler, rf, list(wine.target_names))

with mlflow.start_run(run_name="ex2_custom_pyfunc"):
    sample_output = custom.predict(None, X_test[:3])
    sig = infer_signature(X_test[:3], sample_output)
    mlflow.pyfunc.log_model("model", python_model=custom, signature=sig)
    rid = mlflow.active_run().info.run_id

loaded = mlflow.pyfunc.load_model(f"runs:/{rid}/model")
print(loaded.predict(X_test[:5]))
print("✅ Custom model works!")

---

## 🎓 Module 5 Complete!

**Next up**: Module 6 — MLFlow Projects →